# core

> Assess Ramabana tool routes and build a Urai warm-start history.

In [ ]:
#| default_exp core

In [ ]:
#| export
from dataclasses import dataclass
from pathlib import Path
import json, time, uuid

from fastcore.script import call_parse
from fastcore.xdg import xdg_state_home
from urai import ToolCall, mk_tool_res_msg

In [ ]:
#| export
@dataclass(frozen=True)
class Finding:
    "One tool-route problem in a recorded turn."
    kind: str
    tool: str
    index: int
    message: str

@dataclass(frozen=True)
class Assessment:
    "The route score and findings for one or more turns."
    score: int
    calls: int
    findings: tuple[Finding, ...]

## Read Ramabana history

Ramabana stores completed turns as JSON Lines. Each record contains an `activity` list with the tool name, arguments, result, duration, and status.

In [ ]:
#| export
RAMABANA_HISTORY = Path.home()/'.config/ramabana/agent-history.jsonl'

def read_history(
    path=RAMABANA_HISTORY, # Ramabana `agent-history.jsonl` path
    session=None,         # optional session id
):
    "Read completed Ramabana turns in append order."
    records = []
    with Path(path).expanduser().open() as f:
        for line in f:
            record = json.loads(line)
            if not session or record.get('session') == session: records.append(record)
    return records

## Assess a route

A route loses points for a generic GitHub search before FOSSICK, a malformed notebook edit, a command-line usage error, or a repeated failed action. A real failing test is not a command-line usage error.

In [ ]:
#| export
_RESEARCH_TOOLS = {'web_search', 'read_url', 'search_code', 'run_shell'}

def _is_fossick_repo(action):
    args = action.get('args') or {}
    text = ' '.join(str(v) for v in args.values())
    return action.get('tool') == 'run_shell' and 'fossick read-gh-repo' in text

def assess_turn(turn):
    "Assess the tool route recorded in one Ramabana turn."
    acts = turn.get('activity') or []
    findings = []
    prompt = turn.get('prompt', '').lower()
    if 'github' in prompt and 'fossick' in prompt:
        research = [(i, a) for i, a in enumerate(acts) if a.get('tool') in _RESEARCH_TOOLS]
        if research and not _is_fossick_repo(research[0][1]):
            i, a = research[0]
            findings.append(Finding('route', a.get('tool', ''), i,
                'Use fossick read-gh-repo as the first repository research call.'))
    failed = set()
    for i, a in enumerate(acts):
        if a.get('ok', False): continue
        tool, detail = a.get('tool', ''), str(a.get('detail', ''))
        key = (tool, json.dumps(a.get('args') or {}, sort_keys=True, default=str))
        if key in failed:
            findings.append(Finding('repeat_failure', tool, i,
                'Diagnose or change route before repeating a failed call.'))
        failed.add(key)
        if tool == 'edit_cell' and 'could not parse commands' in detail:
            findings.append(Finding('tool_protocol', tool, i,
                'Use the notebook editor command format from its current tool contract.'))
        if tool == 'run_shell' and ('usage:' in detail or 'unrecognized arguments' in detail):
            findings.append(Finding('tool_protocol', tool, i,
                'Read the project command contract before retrying.'))
    return Assessment(max(0, 100 - 20*len(findings)), len(acts), tuple(findings))

def assess_history(turns):
    "Assess several turns as one route corpus."
    assessments = [assess_turn(t) for t in turns]
    findings = tuple(f for a in assessments for f in a.findings)
    return Assessment(max(0, 100 - 20*len(findings)), sum(a.calls for a in assessments), findings)

The first research call determines whether a repository request took the intended route.

In [ ]:
bad_route = {
    'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
    'activity': [
        {'tool': 'web_search', 'ok': True, 'args': {'query': 'llmdojo'}},
        {'tool': 'run_shell', 'ok': True,
         'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'}},
    ],
}
a = assess_turn(bad_route)
assert a.score == 80 and a.findings[0].kind == 'route'
a

A repeated malformed edit records both the protocol error and the failure to change route.

In [ ]:
bad_edit = {'activity': [
    {'tool': 'edit_cell', 'ok': False, 'args': {'commands': ''},
     'detail': 'ERROR: could not parse commands'},
    {'tool': 'edit_cell', 'ok': False, 'args': {'commands': ''},
     'detail': 'ERROR: could not parse commands'},
]}
a = assess_turn(bad_edit)
assert [f.kind for f in a.findings] == ['tool_protocol', 'repeat_failure', 'tool_protocol']
a

## Build the warm start

`warm_start` returns ordinary Urai history. Rishi re-exports the same chat contract, so any Rishi backend can continue this history through `messages=`.

In [ ]:
#| export
ROUND_REVISION = 1

def warm_start():
    "Return a reviewed tool-routing demonstration as canonical Urai history."
    tc = ToolCall('run_shell', {
        'command': "fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo --globs 'README*,llmdojo/*.py,pyproject.toml' --limit 30",
        'cwd': '.', 'timeout': 120})
    return [
        {'role': 'user', 'content': 'Use FOSSICK to research the AnswerDotAI llmdojo GitHub repository.'},
        {'role': 'assistant', 'content': 'The request names a GitHub repository and FOSSICK, so use its repository reader first.', 'tool_calls': [tc]},
        mk_tool_res_msg(tc, '# llmdojo\nLLM coding agents imitate what their context shows.'),
        {'role': 'assistant', 'content': 'FOSSICK read the repository directly. I will use its source files as the evidence.'},
    ]

def drona_version():
    "The warm-start revision."
    from drona import __version__
    return f'{__version__}:{ROUND_REVISION}'

def receipt_path(state=None):
    "The completion receipt file."
    return Path(state) if state else xdg_state_home()/'drona/completions.json'

def register_completion(state=None):
    "Record one clean round and return its completion id."
    path = receipt_path(state)
    path.parent.mkdir(parents=True, exist_ok=True)
    cid = uuid.uuid4().hex
    records = json.loads(path.read_text()) if path.exists() else {}
    records[cid] = {'version': drona_version(), 'at': time.time()}
    path.write_text(json.dumps(records, indent=2))
    return cid

def completion_valid(cid, state=None):
    "Does `cid` name a completion for the current round?"
    path = receipt_path(state)
    if not cid or not path.exists(): return False
    return (json.loads(path.read_text()).get(cid) or {}).get('version') == drona_version()

def prepare_chat(chat):
    "Prepend the Drona warm start to an empty Urai-compatible chat."
    if chat.hist: raise ValueError('Drona prepares an empty chat only')
    chat.hist = chat.fmt2hist(warm_start())
    if hasattr(chat, '_recreate_conv'): chat._recreate_conv()
    return chat

In [ ]:
hist = warm_start()
assert [m['role'] for m in hist] == ['user', 'assistant', 'tool', 'assistant']
assert hist[1]['tool_calls'][0].name == 'run_shell'
assert hist[2]['tool_call_id'] == hist[1]['tool_calls'][0]['id']
hist

In [ ]:
import tempfile
state = Path(tempfile.mkdtemp())/'completions.json'
cid = register_completion(state)
assert completion_valid(cid, state)
old = json.loads(state.read_text())
old[cid]['version'] = 'old:0'
state.write_text(json.dumps(old))
assert not completion_valid(cid, state)

In [ ]:
#| export
@call_parse
def main(
    history: str=str(RAMABANA_HISTORY), # Ramabana history path
    session: str=None,                  # optional session id
):
    "Assess persisted Ramabana tool routes."
    a = assess_history(read_history(history, session))
    print(json.dumps({'score': a.score, 'calls': a.calls,
                      'findings': [f.__dict__ for f in a.findings]}, indent=2))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()